In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Compute distribution of order counts per customer, excluding comments with special requests
# 1. Filter irrelevant orders and select necessary columns
# 2. Count orders per customer
# 3. Reindex to include all customers (zero for no orders)
# 4. Tabulate distribution and sort

total = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        ["O_CUSTKEY", "O_ORDERKEY"]
    ]
    .groupby("O_CUSTKEY", sort=False)["O_ORDERKEY"]
    .count()
    .reindex(customer["C_CUSTKEY"], fill_value=0)
    .value_counts(sort=False)
    .rename_axis("C_COUNT")
    .reset_index(name="CUSTDIST")
    .sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])
)